# Yotuubef Colab GPU Runner

This notebook runs your repo on Google Colab with GPU support and environment-variable based secrets (so you do not need to keep uploading credential files).

## Before you run
1. In Colab: **Runtime -> Change runtime type -> GPU**
2. Add these in **Colab Secrets** (left sidebar key icon):
   - `REDDIT_CLIENT_ID`
   - `REDDIT_CLIENT_SECRET`
   - `REDDIT_USER_AGENT` (optional, default is set)
   - `GEMINI_API_KEY` (optional; only if you explicitly use Gemini features)
   - `NVIDIA_NIM_API_KEY`
   - `YOUTUBE_TOKEN_JSON` (required only for YouTube upload mode)
   - `BACKGROUND_FOLDER` (optional, defaults to `/content/drive/MyDrive/backround`)
## Workflow modes
- **single**: Process a single Reddit video
- **hybrid**: Run evidence-first documentary workflow with pause/resume (default)


In [ ]:
# Setup system + clone repo + install dependencies
!nvidia-smi
!apt-get -qq update
!apt-get -qq install -y ffmpeg gcc g++ sox

REPO_URL = "https://github.com/beenycool/yotuubef.git"
REPO_DIR = "/content/yotuubef"

import os
if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR

%cd /content/yotuubef
!git pull
!sed -i '/^-r requirements.txt$/d' requirements.txt

!python -m pip install -q --upgrade pip
!python -m pip install -q "numpy<2.1"
!python -m pip install -q -r requirements-ci.txt
!python -m pip install -q git+https://github.com/QwenLM/Qwen3-TTS.git
!python -m pip install -q \
  asyncpraw asyncprawcore \
  python-dotenv PyYAML "pydantic>=2,<3" \
  google-generativeai google-api-python-client google-auth-oauthlib google-auth-httplib2 \
  moviepy imageio-ffmpeg opencv-python-headless pillow \
  "yt-dlp>=2024.1.0" scipy librosa soundfile \
  psutil gputil pynvml nest_asyncio
!python -m pip install -q "numpy<2.1" "pydantic>=2.11,<3" "aiosqlite==0.17.0" "ruff>=0.9.3"


In [ ]:
# Load secrets from Colab Secrets into environment variables
import os

try:
    from google.colab import userdata

    def get_secret(name, default=""):
        try:
            value = userdata.get(name)
            return value if value is not None else default
        except Exception:
            return default
except Exception:
    def get_secret(name, default=""):
        return os.getenv(name, default)

os.environ["REDDIT_CLIENT_ID"] = get_secret("REDDIT_CLIENT_ID")
os.environ["REDDIT_CLIENT_SECRET"] = get_secret("REDDIT_CLIENT_SECRET")
os.environ["REDDIT_USER_AGENT"] = get_secret("REDDIT_USER_AGENT", "python:VideoBot:v2.0 (by /u/yourusername)")
os.environ["GEMINI_API_KEY"] = get_secret("GEMINI_API_KEY")
os.environ["NVIDIA_NIM_API_KEY"] = get_secret("NVIDIA_NIM_API_KEY")

# Optional, only needed if RUN_FULL_UPLOAD is True
youtube_token_json = get_secret("YOUTUBE_TOKEN_JSON", "")
if youtube_token_json:
    os.environ["YOUTUBE_TOKEN_JSON"] = youtube_token_json

background_folder = get_secret("BACKGROUND_FOLDER", "")
if background_folder:
    os.environ["BACKGROUND_FOLDER"] = background_folder

required = ["REDDIT_CLIENT_ID", "REDDIT_CLIENT_SECRET", "NVIDIA_NIM_API_KEY"]
missing = [key for key in required if not os.environ.get(key)]
if missing:
    raise ValueError(f"Missing required Colab Secrets: {missing}")

print("Environment variables are set.")
print("YOUTUBE_TOKEN_JSON found:", bool(os.environ.get("YOUTUBE_TOKEN_JSON")))
print("BACKGROUND_FOLDER preset:", os.environ.get("BACKGROUND_FOLDER", "") or "(not set)")


In [ ]:
# Mount Google Drive and set default background folder
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    print("Not running in Colab; Google Drive mount skipped.")

default_bg_folder = '/content/drive/MyDrive/backround'
if not os.environ.get('BACKGROUND_FOLDER'):
    os.environ['BACKGROUND_FOLDER'] = default_bg_folder

bg_folder = Path(os.environ['BACKGROUND_FOLDER'])
bg_files = sorted(bg_folder.glob('*.mp4')) if bg_folder.exists() else []
print('BACKGROUND_FOLDER:', bg_folder)
print('Background .mp4 count:', len(bg_files))
if not bg_files:
    print('No .mp4 files found. Add videos to this folder or set BACKGROUND_FOLDER secret.')


In [ ]:
# Runtime options
WORKFLOW_MODE = "hybrid"  # "single" or "hybrid"

# For single mode:
REDDIT_URL = "https://www.reddit.com/r/oddlysatisfying/comments/<post_id>/"  # change this
RUN_FULL_UPLOAD = True  # True: runs main.py single and uploads to YouTube, False: local render only

# For hybrid mode:
HYBRID_PROJECT = "my_project"  # Project name for hybrid workflow
HYBRID_RESUME = False  # Resume from saved state?
HYBRID_PHASE = None  # Optional: override phase (e.g., "evidence_gathering")
HYBRID_NO_UPLOAD = False  # Disable upload stage for hybrid

if RUN_FULL_UPLOAD and not os.environ.get("YOUTUBE_TOKEN_JSON"):
    raise ValueError("RUN_FULL_UPLOAD=True requires YOUTUBE_TOKEN_JSON in Colab Secrets")

print("Workflow mode:", WORKFLOW_MODE)
print("Configured URL:", REDDIT_URL)
print("Upload mode:", RUN_FULL_UPLOAD)


In [ ]:
# Run pipeline
import asyncio
import json
import subprocess
import shlex
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()

def ffmpeg_has_nvenc() -> bool:
    try:
        out = subprocess.check_output(["ffmpeg", "-hide_banner", "-encoders"], stderr=subprocess.STDOUT, text=True)
        return "h264_nvenc" in out
    except Exception:
        return False

# Build command based on workflow mode
if WORKFLOW_MODE == "single":
    if RUN_FULL_UPLOAD:
        cmd = f"python3 main.py single {shlex.quote(REDDIT_URL)}"
        print("Running:", cmd)
        proc = subprocess.run(cmd, shell=True, text=True)
        if proc.returncode != 0:
            raise RuntimeError(f"Command failed with exit code {proc.returncode}")
    else:
        from src.config.settings import get_config
        from src.enhanced_orchestrator import EnhancedVideoOrchestrator

        cfg = get_config()
        if not ffmpeg_has_nvenc():
            cfg.video.enable_speed_optimization = False
            cfg.video.video_codec_cpu = "libx264"
            cfg.video.ffmpeg_cpu_preset = "ultrafast"
            print("NVENC not available: using CPU x264 fallback.")
        else:
            print("NVENC available: GPU encoding can be used.")

        async def run_local_pipeline(reddit_url: str):
            orchestrator = EnhancedVideoOrchestrator()

            download_result = await orchestrator._download_and_analyze_video(reddit_url)
            if not download_result.get("success"):
                return download_result

            video_path = download_result["video_path"]
            analysis = download_result["analysis"]

            analysis = await orchestrator._perform_enhanced_analysis(video_path, analysis)
            if orchestrator.enable_cinematic_editing:
                analysis = await orchestrator._apply_cinematic_analysis(video_path, analysis)

            process_result = await orchestrator._process_with_enhancements(video_path, analysis, options={})
            if not process_result.get("success"):
                return {"success": False, "stage": "process", "details": process_result}

            thumbnail_results = await orchestrator._generate_ab_test_thumbnails(video_path, analysis, process_result)

            result = {
                "success": True,
                "source_video": str(video_path),
                "processed_video": str(process_result.get("video_path")),
                "processing_stats": process_result.get("processing_stats", {}),
                "thumbnail_variants": thumbnail_results,
            }

            out_dir = Path("data/results")
            out_dir.mkdir(parents=True, exist_ok=True)
            (out_dir / "colab_run_result.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
            return result

        local_result = await run_local_pipeline(REDDIT_URL)
        print(json.dumps(local_result, indent=2))

elif WORKFLOW_MODE == "hybrid":
    cmd_parts = ["python3", "main.py", "hybrid", HYBRID_PROJECT]
    if HYBRID_RESUME:
        cmd_parts.append("--resume")
    if HYBRID_PHASE:
        cmd_parts.extend(["--phase", HYBRID_PHASE])
    if HYBRID_NO_UPLOAD:
        cmd_parts.append("--no-upload")
    cmd = " ".join(shlex.quote(str(p)) for p in cmd_parts)
    print("Running:", cmd)
    proc = subprocess.run(cmd, shell=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}")

else:
    raise ValueError(f"Unknown WORKFLOW_MODE: {WORKFLOW_MODE}")


In [ ]:
# Show latest result file (if any)
from pathlib import Path

results_dir = Path("data/results")
if results_dir.exists():
    files = sorted(results_dir.glob("*.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    if files:
        print("Latest result:", files[0])
        print(files[0].read_text(encoding="utf-8")[:4000])
    else:
        print("No JSON result files found yet.")
else:
    print("Results directory does not exist yet.")

# Check for hybrid project outputs
hybrid_projects_dir = Path("data/hybrid_projects")
if hybrid_projects_dir.exists():
    project_dirs = sorted(hybrid_projects_dir.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True)
    if project_dirs:
        latest_project = project_dirs[0]
        print(f"\nLatest hybrid project: {latest_project.name}")
        state_file = latest_project / "state.json"
        if state_file.exists():
            print("State:", state_file.read_text(encoding="utf-8")[:2000])
        artifacts = list(latest_project.glob("*"))
        print(f"Artifacts ({len(artifacts)}):", [a.name for a in artifacts[:10]])


In [ ]:
# Optional: download latest processed video from Colab runtime
from pathlib import Path
from google.colab import files

processed_dir = Path("processed")
if processed_dir.exists():
    videos = sorted(processed_dir.glob("*.mp4"), key=lambda p: p.stat().st_mtime, reverse=True)
    if videos:
        print("Downloading:", videos[0])
        files.download(str(videos[0]))
    else:
        print("No processed .mp4 files found.")
else:
    print("Processed directory does not exist yet.")
